# Preprocessing & Feature Engineering

This notebook prepares the diabetic dataset for machine learning by:
- Inspecting data quality
- Checking missing values
- Dropping low-value columns
- Handling '?' placeholders
- Encoding categorical variables
- Scaling numeric features
- Creating train/test splits


In [1]:
# Set working directory and load dataset

import os
import pandas as pd
import numpy as np

In [2]:
# Set working directory to project root
os.chdir(r"d:\NHS-ML-Project\hospital-readmission-ml-and-nhs-ae-dashboard")

In [3]:
# Load the raw diabetic dataset
df = pd.read_csv("data/ml_raw/diabetic_data.csv")

# Preview the first few rows
df.head()


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Check Dataset Structure

We inspect the shape and data types to understand the dataset before
performing any cleaning or preprocessing.


In [4]:
#  Check shape and data types

# Number of rows and columns
df.shape


# Data types and non-null counts
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## Check Missing Values

We calculate missing values for each column to identify features with
high missingness that may need to be dropped or imputed.


In [5]:
#  Check missing values per column

# Count missing values
missing_counts = df.isna().sum()

# Percentage of missing values
missing_percent = (missing_counts / len(df)) * 100

# Combine into a table
missing_table = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percent': missing_percent
})

# Sort by highest missing percentage
missing_table.sort_values(by='missing_percent', ascending=False)


,missing_count,missing_percent
max_glu_serum,96420,94.746772
A1Cresult,84748,83.277322
race,0,0.000000
gender,0,0.000000
age,0,0.000000
weight,0,0.000000
admission_type_id,0,0.000000
discharge_disposition_id,0,0.000000
admission_source_id,0,0.000000
time_in_hospital,0,0.000000


## Identify '?' as Missing Values
The dataset uses '?' as a placeholder for missing values in several
categorical columns. We count how often '?' appears in each column.


In [6]:
#  Count '?' values in each column

question_mark_counts = (df == "?").sum()

# Show only columns that contain '?'
question_mark_counts[question_mark_counts > 0].sort_values(ascending=False)


weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

##  Drop Columns with Extremely High Missing Values

Some columns in the dataset contain extremely high proportions of missing values 
or placeholder values ('?'). These columns do not provide useful information for 
modelling and can negatively impact data quality. 

We remove the following columns (only if they exist in the dataset):
- max_glu_serum
- A1Cresult
- examide
- citoglipton

This helps simplify the dataset and improves the reliability of downstream 
preprocessing and modelling steps.


In [7]:
# Drop high-missing columns ONLY if they exist

cols_to_drop = ['max_glu_serum', 'A1Cresult', 'examide', 'citoglipton']

# Keep only columns that are present in the dataframe
cols_to_drop = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_to_drop)

df.shape


(101766, 46)

##  Replace '?' With NaN

The dataset uses the character '?' to represent missing values instead of 
standard NaN. To ensure proper handling of missing data, we replace all '?' 
entries with actual NaN values. This allows pandas and scikit-learn to treat 
them correctly during preprocessing.


In [8]:
# ---------------------------------------------
#  Replace '?' with NaN
# ---------------------------------------------

df = df.replace("?", np.nan)

# Check missing values again
df.isna().sum().sort_values(ascending=False).head(20)


weight                      98569
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
age                             0
admission_type_id               0
discharge_disposition_id        0
time_in_hospital                0
gender                          0
patient_nbr                     0
encounter_id                    0
num_procedures                  0
num_lab_procedures              0
admission_source_id             0
num_medications                 0
number_inpatient                0
number_emergency                0
dtype: int64

##  Create Binary Target Column (readmitted_30)

The dataset contains the column `readmitted`, not `readmitted_30`.  
We convert it into a binary target:

- <30 → 1  
- NO  → 0  
- >30 → 0  

This creates the correct target variable for modelling.


In [9]:
# ---------------------------------------------
# Create binary target column
# ---------------------------------------------

df['readmitted_30'] = df['readmitted'].map({
    '<30': 1,
    'NO': 0,
    '>30': 0
})

# Drop the original readmitted column
df = df.drop(columns=['readmitted'])

# Ensure target is numeric so it does NOT get one-hot encoded
df['readmitted_30'] = df['readmitted_30'].astype(int)

# Confirm
df['readmitted_30'].value_counts()


readmitted_30
0    90409
1    11357
Name: count, dtype: int64

## Fill Missing Values

After converting '?' to NaN, some categorical columns still contain missing values.
We fill these with 'Unknown' to avoid issues during encoding and model training.


In [10]:
#  Fill missing values in categorical columns

# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Fill missing values with 'Unknown'
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Confirm no missing values remain in categorical columns
df[cat_cols].isna().sum().sum()


np.int64(0)

## Convert Categorical Columns to Category Type

Converting object columns to 'category' reduces memory usage and speeds up
encoding during preprocessing.


In [11]:
# Convert object columns to 'category' dtype

for col in cat_cols:
    df[col] = df[col].astype('category')

# Check updated data types
df.dtypes.head(20)


encounter_id                   int64
patient_nbr                    int64
race                        category
gender                      category
age                         category
weight                      category
admission_type_id              int64
discharge_disposition_id       int64
admission_source_id            int64
time_in_hospital               int64
payer_code                  category
medical_specialty           category
num_lab_procedures             int64
num_procedures                 int64
num_medications                int64
number_outpatient              int64
number_emergency               int64
number_inpatient               int64
diag_1                      category
diag_2                      category
dtype: object

##  Convert Categorical Columns to Category Type

Converting object columns to category dtype reduces memory usage and speeds up 
encoding in the next step. This is especially useful for large healthcare 
datasets with many categorical variables.


In [12]:
# Identify categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Fill missing values with 'Unknown'
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Convert object columns to 'category' dtype
for col in cat_cols:
    df[col] = df[col].astype('category')


## One-Hot Encode Categorical Variables

Machine learning models require numerical inputs. We convert categorical 
variables into numeric dummy variables using one-hot encoding. We drop the 
first category to avoid multicollinearity.


In [13]:
# ---------------------------------------------
#  One-hot encode categorical variables
# ---------------\------------------------------

df_encoded = pd.get_dummies(df, drop_first=True)

# Check encoded shape
df_encoded.shape


(101766, 2431)

In [14]:
# Clean column names for XGBoost compatibility
df_encoded.columns = (
    df_encoded.columns
    .str.replace('[', '(', regex=False)
    .str.replace(']', ')', regex=False)
    .str.replace('<', 'lt_', regex=False)
    .str.replace('>', 'gt_', regex=False)
)


## Scale Numeric Features

We scale only the numeric feature columns (excluding the target) using 
StandardScaler. This ensures all numeric features are on a similar scale, 
which improves model performance for algorithms like Logistic Regression.


In [15]:
# ---------------------------------------------
# Scale numeric features
# ---------------------------------------------

from sklearn.preprocessing import StandardScaler

# Identify numeric columns EXCEPT the target
num_cols = df_encoded.drop(columns=['readmitted_30']).select_dtypes(include=['int64','float64']).columns

# Initialize scaler
scaler = StandardScaler()

# Scale numeric columns
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])


Remove ALL model training from this notebook

In [19]:
import os

os.makedirs("../data/processed", exist_ok=True)
df_encoded.to_csv("../data/processed/processed_diabetic_data.csv", index=False)
